# 03b — Transformer fine-tuning: DistilBERT sentiment classifier

Supervised binary sentiment (positive vs negative) with `distilbert-base-uncased`, compared with the TF-IDF + Linear SVM from notebook 03 on the **same 72,846-review held-out test set**.

**Scope.** MiniLM (`all-MiniLM-L6-v2`) is *not* fine-tuned anywhere in this project. It stays a pretrained embedding model for retrieval (notebook 04). DistilBERT is a separate supervised classifier.

**Rules**
- Hyperparameters are chosen on a **validation set carved from the training partition only**. The test set is never used for selection.
- Selection rule (fixed before looking at results): highest validation Macro-F1; differences under 0.003 count as ties (repeat runs of one configuration varied by about 0.003); ties go to higher negative recall, then the cheaper setting.
- Every reported number comes from a saved result file or a live output.

**How to run:** `Run All`. With `RUN_TRAINING = False` it only loads saved results and models. Set it to `True` to retrain (~35 min for the experiments, ~27 min for the final model on an RTX 3060 Ti).

In [ ]:
import gc, json, os, time
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn
import datasets, transformers
from datasets import Dataset as HFDataset
from scipy.optimize import minimize_scalar
from sklearn.metrics import (accuracy_score, brier_score_loss, classification_report, confusion_matrix,
                             precision_recall_fscore_support, roc_auc_score)
from sklearn.model_selection import train_test_split
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
                          EarlyStoppingCallback, Trainer, TrainingArguments)

SEED, MODEL_NAME, MAX_LEN = 42, "distilbert-base-uncased", 128
RUN_TRAINING = False
RUN_MANUAL_WEIGHT_EXPERIMENTS = False            # True = retrain; False = load saved results/models
ID2LABEL, LABEL2ID = {0: "negative", 1: "positive"}, {"negative": 0, "positive": 1}

def find_root():
    here = Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / "data" / "processed").exists():
            return d
    raise FileNotFoundError("Run from inside the project (a folder containing data/processed).")

ROOT = find_root()
DATA_CSV = ROOT / "data" / "processed" / "reviews_cleaned.csv"
SPLIT_FILE = ROOT / "data" / "processed" / "split_indices.npz"     # written by notebook 03
ART = ROOT / "notebooks" / "artifacts"          # single place to change when files move to models/ and reports/
RUNS = ROOT / "scratch" / "runs"                # training checkpoints (not deliverables; keep out of Git)
FINAL_DIR = ART / "distilbert_full281k"
for d in (ART, RUNS):
    d.mkdir(parents=True, exist_ok=True)

print(f"torch {torch.__version__} | transformers {transformers.__version__} | datasets {datasets.__version__}")
print("CUDA:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")
if RUN_TRAINING:
    assert torch.cuda.is_available(), "Training needs the GPU (CUDA build of PyTorch)."
# Results manifest for 50k runs and final model
RESULTS_JSON = ART / 'distilbert_results.json'
results = json.loads(RESULTS_JSON.read_text()) if RESULTS_JSON.exists() else []


torch 2.14.0+cu126 | transformers 5.16.1 | datasets 5.0.1
CUDA: True | NVIDIA GeForce RTX 3060 Ti


## 1. Data subsets and leakage checks

| Set | Size | Use |
|---|---|---|
| Transformer-train subset | 50,000 | hyperparameter experiments |
| Validation | 9,997 | epoch / hyperparameter selection (from the training partition) |
| Full training set | 281,271 | final model (all training reviews not in validation) |
| Test | 72,846 | final evaluation only |

Loaded from saved files when they exist; otherwise rebuilt from the saved split indices with `random_state=42`. Validation reviews whose exact text also appears in the training data are removed from validation, and every copy of validation text is removed from the full training set.

In [3]:
FILES = {k: ART / f"{k}.pkl" for k in ("tr_sub_50k", "val_sub_9997", "full_train_281271", "test_72846")}
EXPECTED = {"tr_sub_50k": 50_000, "val_sub_9997": 9_997, "full_train_281271": 281_271, "test_72846": 72_846}

if all(p.exists() for p in FILES.values()):
    tr50, val, full_tr, test = (pd.read_pickle(FILES[k]) for k in FILES)
    print("Loaded saved subsets.")
else:
    binary = pd.read_csv(DATA_CSV)
    binary = binary[binary["sentiment_label"] != "neutral"]
    split = np.load(SPLIT_FILE)
    def part(index):
        t = binary.loc[index]
        return pd.DataFrame({"text": t["full_text"].fillna("").astype(str).values,
                             "label": t["sentiment_label"].map(LABEL2ID).values})
    train_all, test = part(split["train_index"]), part(split["test_index"])
    tr50, val = train_test_split(train_all, train_size=50_000, test_size=10_000,
                                 stratify=train_all["label"], random_state=SEED)
    val = val[~val["text"].isin(set(tr50["text"]))]                  # keep validation text-disjoint from train
    full_tr = train_all[~train_all["text"].isin(set(val["text"]))]
    for key, frame in zip(FILES, (tr50, val, full_tr, test)):
        frame.reset_index(drop=True).to_pickle(FILES[key])
    print("Rebuilt subsets from the saved split and cached them.")

tr50, val, full_tr, test = (d.reset_index(drop=True) for d in (tr50, val, full_tr, test))
for key, frame in zip(FILES, (tr50, val, full_tr, test)):
    assert len(frame) == EXPECTED[key], f"{key}: {len(frame):,} rows, expected {EXPECTED[key]:,}"
val_text, test_text = set(val["text"]), set(test["text"])
assert not (set(tr50["text"]) & val_text) and not (set(full_tr["text"]) & val_text), "train/validation text overlap"
assert not (set(full_tr["text"]) & test_text) and not (val_text & test_text), "train or validation overlaps test"

for name, frame in [("train 50k", tr50), ("validation", val), ("full train", full_tr), ("test", test)]:
    print(f"{name:11s} {len(frame):>8,} rows | positive {frame['label'].mean() * 100:.2f}%")
print("Exact-text overlap train/validation, train/test, validation/test: 0 / 0 / 0")

Loaded saved subsets.
train 50k     50,000 rows | positive 84.30%
validation     9,997 rows | positive 84.31%
full train   281,271 rows | positive 84.30%
test          72,846 rows | positive 84.39%
Exact-text overlap train/validation, train/test, validation/test: 0 / 0 / 0


## 2. Tokenization

`max_length = 128` follows a published DistilBERT project on this dataset and keeps training feasible on an 8 GB GPU. Reviews are truncated to 128 tokens and padded per batch (dynamic padding). The statistic below shows how many reviews reach the cap.

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
collator = DataCollatorWithPadding(tokenizer)

def tokenize(frame):
    ds = HFDataset.from_pandas(frame.reset_index(drop=True))
    return ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=MAX_LEN),
                  batched=True, batch_size=2000, remove_columns=["text"])

tok_50, tok_val = tokenize(tr50), tokenize(val)
lens = np.array([len(ids) for ids in tok_50["input_ids"]])
print(f"Token length after truncation: mean {lens.mean():.1f}, median {np.median(lens):.0f}")
print(f"Reviews at the {MAX_LEN}-token cap (truncated or exactly {MAX_LEN}): {(lens >= MAX_LEN).mean() * 100:.1f}%")

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Map:   0%|          | 0/9997 [00:00<?, ? examples/s]

Token length after truncation: mean 82.9, median 79
Reviews at the 128-token cap (truncated or exactly 128): 26.7%


## 3. Training helpers

One function trains any configuration. `weighted=True` uses inverse-frequency class weights in the loss (the same formula as scikit-learn's `class_weight="balanced"`); `patience` adds early stopping on validation Macro-F1. Mixed precision (`fp16`) is used for speed.

In [5]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    p, r, f, _ = precision_recall_fscore_support(labels, preds, labels=[0, 1], zero_division=0)
    return {"accuracy": accuracy_score(labels, preds), "macro_f1": float(f.mean()),
            "neg_precision": p[0], "neg_recall": r[0], "neg_f1": f[0],
            "pos_precision": p[1], "pos_recall": r[1], "pos_f1": f[1]}

class WeightedTrainer(Trainer):
    """Cross-entropy with class weights; everything else is the standard Trainer."""
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss_fn = nn.CrossEntropyLoss(weight=self.class_weights.to(outputs.logits.device))
        loss = loss_fn(outputs.logits.float(), labels)
        return (loss, outputs) if return_outputs else loss

def summarize(trainer, tag, change, lr, bs, ep, seed, minutes):
    evals = [e for e in trainer.state.log_history if "eval_loss" in e]
    best = max(evals, key=lambda e: e["eval_macro_f1"])               # selection metric: validation Macro-F1
    return {"run": tag, "change": change, "lr": lr, "batch": bs, "epochs": ep, "seed": seed,
            "minutes": round(minutes, 1), "best_epoch": int(round(best["epoch"])),
            "val_macro_f1": round(best["eval_macro_f1"], 4), "val_acc": round(best["eval_accuracy"], 4),
            "neg_recall": round(best["eval_neg_recall"], 4), "neg_f1": round(best["eval_neg_f1"], 4),
            "val_loss_by_epoch": [round(e["eval_loss"], 4) for e in evals]}

def train_run(tag, change, lr=2e-5, bs=16, ep=2, seed=SEED, weighted=False, patience=None,
              train_ds=None, save_to=None):
    train_ds = tok_50 if train_ds is None else train_ds
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2, id2label=ID2LABEL, label2id=LABEL2ID)
    args = TrainingArguments(
        output_dir=str(RUNS / tag), eval_strategy="epoch", save_strategy="epoch", save_total_limit=2,
        learning_rate=lr, weight_decay=0.01, per_device_train_batch_size=bs, per_device_eval_batch_size=32,
        num_train_epochs=ep, fp16=True, load_best_model_at_end=True, metric_for_best_model="macro_f1",
        greater_is_better=True, logging_steps=500, report_to="none", seed=seed)
    kwargs = dict(model=model, args=args, train_dataset=train_ds, eval_dataset=tok_val,
                  processing_class=tokenizer, data_collator=collator, compute_metrics=compute_metrics)
    if patience:
        kwargs["callbacks"] = [EarlyStoppingCallback(early_stopping_patience=patience)]
    if weighted:
        counts = np.bincount(np.asarray(train_ds["label"]), minlength=2)
        w = torch.tensor(len(train_ds) / (2 * counts), dtype=torch.float32)
        print("Class weights [negative, positive]:", np.round(w.numpy(), 4))
        trainer = WeightedTrainer(class_weights=w, **kwargs)
    else:
        trainer = Trainer(**kwargs)
    t0 = time.time(); trainer.train(); minutes = (time.time() - t0) / 60
    row = summarize(trainer, tag, change, lr, bs, ep, seed, minutes)
    if save_to is not None:
        trainer.save_model(str(save_to)); tokenizer.save_pretrained(str(save_to))
    del trainer, model; gc.collect(); torch.cuda.empty_cache()
    return row

## 4. Final model: train on the full training set

The settings above were tuned on the 50k subset and carried over, not re-tuned. Training uses all 281,271 training reviews not in validation; the epoch is picked on validation Macro-F1. The 50k run was the first full candidate; the rule for adopting the larger run was fixed beforehand: adopt it only if its validation Macro-F1 beats the 50k runs by more than 0.003.

In [8]:
FINAL_TAG = "full281k_lr2e-5_bs16_ep2"
if RUN_TRAINING and FINAL_TAG not in {r["run"] for r in results}:
    results.append(train_run(FINAL_TAG, "train on 281,271 reviews", train_ds=tokenize(full_tr), save_to=FINAL_DIR))
    RESULTS_JSON.write_text(json.dumps(results, indent=2))
assert FINAL_DIR.exists(), "Final model not found. Set RUN_TRAINING = True to train it."

final = next(r for r in results if r["run"] == FINAL_TAG)
ref = [r["val_macro_f1"] for r in results if r["run"] in ("run_ep2", "s123_lr2e-5_bs16")]
print("Final model validation:", {k: final[k] for k in ("best_epoch", "val_macro_f1", "val_acc", "neg_recall", "minutes", "val_loss_by_epoch")})
print(f"50k-subset runs at the same setting: {ref} (mean {np.mean(ref):.4f}); margin {final['val_macro_f1'] - np.mean(ref):+.4f} (rule: > 0.003)")

Final model validation: {'best_epoch': 2, 'val_macro_f1': 0.9559, 'val_acc': 0.9766, 'neg_recall': 0.9293, 'minutes': 27.1, 'val_loss_by_epoch': [0.0771, 0.0876]}
50k-subset runs at the same setting: [0.9443, 0.9423] (mean 0.9433); margin +0.0126 (rule: > 0.003)


## 5. Held-out test evaluation (final model, reporting only)

The final model's logits on the test set are computed once and cached. Nothing is changed after seeing these numbers.

In [9]:
def predict_logits(frame, model_dir):
    tok = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir)
    ds = HFDataset.from_pandas(frame.reset_index(drop=True)).map(
        lambda b: tok(b["text"], truncation=True, max_length=MAX_LEN),
        batched=True, batch_size=2000, remove_columns=["text"])
    runner = Trainer(model=model, data_collator=DataCollatorWithPadding(tok),
                     args=TrainingArguments(output_dir=str(RUNS / "eval"), per_device_eval_batch_size=64,
                                            fp16=torch.cuda.is_available(), report_to="none"))
    out = runner.predict(ds)
    return out.predictions.astype(np.float64), out.label_ids

LOGITS_FILE = ART / "distilbert281k_test_logits.npy"
if LOGITS_FILE.exists():
    test_logits = np.load(LOGITS_FILE); y_test = test["label"].to_numpy()
else:
    test_logits, y_test = predict_logits(test, FINAL_DIR)
    np.save(LOGITS_FILE, test_logits)
assert len(test_logits) == len(y_test) == EXPECTED["test_72846"]

def softmax(z, T=1.0):
    z = z / T; z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z); return e / e.sum(axis=1, keepdims=True)

probs = softmax(test_logits)[:, 1]
y_pred = test_logits.argmax(axis=1)
cm = confusion_matrix(y_test, y_pred); tn, fp, fn, tp = (int(v) for v in cm.ravel())
p, r, f, _ = precision_recall_fscore_support(y_test, y_pred, labels=[0, 1])
live = {"train": len(full_tr), "accuracy": accuracy_score(y_test, y_pred), "macro_f1": float(f.mean()),
        "roc_auc": roc_auc_score(y_test, probs), "neg_precision": p[0], "neg_recall": r[0], "neg_f1": f[0],
        "pos_f1": f[1], "TN": tn, "FP": fp, "FN": fn, "TP": tp}
np.save(ART / "distilbert281k_test_probs.npy", probs)

print(classification_report(y_test, y_pred, target_names=["negative", "positive"], digits=4))
print("ROC-AUC: %.4f" % live["roc_auc"]); print(f"TN={tn:,}  FP={fp:,} | FN={fn:,}  TP={tp:,}")

              precision    recall  f1-score   support

    negative     0.9186    0.9138    0.9162     11372
    positive     0.9841    0.9850    0.9845     61474

    accuracy                         0.9739     72846
   macro avg     0.9513    0.9494    0.9504     72846
weighted avg     0.9739    0.9739    0.9739     72846

ROC-AUC: 0.9930
TN=10,392  FP=980 | FN=921  TP=60,553


## 6. Comparison on the same test set

SVM and the 50k DistilBERT rows come from `final_test_results.json` (recorded from their original evaluation outputs); the 281k row is the live result above. The SVM was trained on all 291,276 training reviews, the 50k model on 50,000, the final model on 281,271 — state this in the report.

In [10]:
recorded = json.loads((ART / "final_test_results.json").read_text())
rows = {"TF-IDF + Linear SVM": recorded["svm_tfidf_linearsvc_C0.5_cwNone"],
        "DistilBERT (50k train)": recorded["distilbert_50k"],
        "DistilBERT (281k train) — final": live}
keys = ["train", "accuracy", "macro_f1", "roc_auc", "neg_precision", "neg_recall", "neg_f1", "pos_f1", "FP", "FN"]
table = pd.DataFrame({name: {k: d[k] for k in keys} for name, d in rows.items()}).T
table["errors"] = table["FP"] + table["FN"]
print(table.round(4).to_string())

rec = recorded["distilbert_281k_FINAL"]
print("\nReproducibility check, final model (this run vs recorded):")
for k in ("accuracy", "macro_f1", "roc_auc", "neg_recall"):
    print(f"  {k:10s} {live[k]:.4f} vs {rec[k]:.4f}  (diff {live[k] - rec[k]:+.4f})")

                                    train  accuracy  macro_f1  roc_auc  neg_precision  neg_recall  neg_f1  pos_f1      FP      FN  errors
TF-IDF + Linear SVM              291276.0    0.9617    0.9253   0.9854         0.9040      0.8444  0.8732  0.9775  1769.0  1020.0  2789.0
DistilBERT (50k train)            50000.0    0.9673    0.9375   0.9897         0.9037      0.8851  0.8943  0.9807  1307.0  1073.0  2380.0
DistilBERT (281k train) — final  281271.0    0.9739    0.9504   0.9930         0.9186      0.9138  0.9162  0.9845   980.0   921.0  1901.0



Reproducibility check, final model (this run vs recorded):
  accuracy   0.9739 vs 0.9739  (diff +0.0000)
  macro_f1   0.9504 vs 0.9504  (diff -0.0000)
  roc_auc    0.9930 vs 0.9930  (diff +0.0000)
  neg_recall 0.9138 vs 0.9138  (diff +0.0000)


## 7. Probability calibration

Raw softmax probabilities can be overconfident (validation loss rose with more epochs). A single temperature `T` is fitted on the **validation set only** and applied as `softmax(logits / T)`. Dividing by T never changes which class wins, so accuracy and Macro-F1 are unchanged; only the confidence changes. Test data is used only to report ECE and Brier score.

In [11]:
val_logits, y_val = predict_logits(val, FINAL_DIR)

def nll(T, z, y):
    return -np.log(np.clip(softmax(z, T)[np.arange(len(y)), y], 1e-12, None)).mean()

def ece(p_pos, y, bins=10):
    conf, pred = np.maximum(p_pos, 1 - p_pos), (p_pos >= 0.5).astype(int)
    edges, total = np.linspace(0.5, 1.0, bins + 1), 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            total += m.mean() * abs((pred[m] == y[m]).mean() - conf[m].mean())
    return total

T = float(minimize_scalar(lambda t: nll(t, val_logits, y_val), bounds=(0.5, 5.0), method="bounded").x)
print(f"Fitted temperature T = {T:.3f} (T > 1 means the raw model was overconfident)")
print("\n                 ECE     Brier   (lower is better)")
for name, z, y, t in [("validation raw", val_logits, y_val, 1.0), ("validation cal.", val_logits, y_val, T),
                      ("test raw", test_logits, y_test, 1.0), ("test calibrated", test_logits, y_test, T)]:
    pp = softmax(z, t)[:, 1]
    print(f"{name:16s} {ece(pp, y):.4f}  {brier_score_loss(y, pp):.4f}")
assert (softmax(test_logits).argmax(1) == softmax(test_logits, T).argmax(1)).all()
print("\nPredictions identical before/after calibration: True")

np.save(ART / "temperature_T.npy", np.array([T]))
(ART / "label_map.json").write_text(json.dumps({"id2label": ID2LABEL, "label2id": LABEL2ID}, indent=2))
print("Saved temperature_T.npy and label_map.json to", ART.relative_to(ROOT))

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Map:   0%|          | 0/9997 [00:00<?, ? examples/s]

Fitted temperature T = 1.654 (T > 1 means the raw model was overconfident)

                 ECE     Brier   (lower is better)
validation raw   0.0180  0.0207
validation cal.  0.0099  0.0189
test raw         0.0201  0.0229
test calibrated  0.0094  0.0208

Predictions identical before/after calibration: True
Saved temperature_T.npy and label_map.json to notebooks\artifacts


## Limitations (state these in the report)

- Hyperparameters were tuned on a 50k subset and carried to the 281k run without re-tuning.
- Most configurations were run with one seed; run-to-run variation was about 0.003 Macro-F1 on validation.
- SVM and DistilBERT are the best of each family within a small time budget, not a proof that one family is always better.
- Inputs are truncated to 128 tokens; a fraction of long reviews lose their tail (see section 2).
- Binary model: neutral (3-star) reviews were not in training. Calibration was fitted on this dataset only.
- Star rating is a noisy proxy for sentiment. Dataset is old and food-specific; this is a project model, not a production system.

In [15]:
# Manual class-weight experiments
# Order = [negative, positive]

weights_to_test = {
    "unweighted": [1.0, 1.0],
    "1.5x_negative": [1.5, 1.0],
    "2x_negative": [2.0, 1.0],
    "3x_negative": [3.0, 1.0],
}

for name, weights in weights_to_test.items():
    print(f"{name:18s} -> negative: {weights[0]:.1f}, positive: {weights[1]:.1f}")

unweighted         -> negative: 1.0, positive: 1.0
1.5x_negative      -> negative: 1.5, positive: 1.0
2x_negative        -> negative: 2.0, positive: 1.0
3x_negative        -> negative: 3.0, positive: 1.0


In [16]:
def train_manual_weight_run(tag, negative_weight, positive_weight=1.0, seed=SEED):
    """
    Train DistilBERT with manually specified class weights.
    Class order is [negative, positive].
    Uses the existing 50k training subset and 9,997 validation set.
    """

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2,
        id2label=ID2LABEL,
        label2id=LABEL2ID
    )

    args = TrainingArguments(
        output_dir=str(RUNS / tag),
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        learning_rate=2e-5,
        weight_decay=0.01,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        num_train_epochs=2,
        fp16=True,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        logging_steps=500,
        report_to="none",
        seed=seed
    )

    weights = torch.tensor(
        [negative_weight, positive_weight],
        dtype=torch.float32
    )

    print(f"\n=== {tag} ===")
    print("Class weights [negative, positive]:", weights.tolist())

    trainer = WeightedTrainer(
        model=model,
        args=args,
        train_dataset=tok_50,
        eval_dataset=tok_val,
        processing_class=tokenizer,
        data_collator=collator,
        compute_metrics=compute_metrics,
        class_weights=weights
    )

    t0 = time.time()
    trainer.train()
    minutes = (time.time() - t0) / 60

    result = summarize(
        trainer,
        tag,
        f"manual [{negative_weight}:1]",
        2e-5,
        16,
        2,
        seed,
        minutes
    )

    print("\nValidation result:")
    print(f"Macro-F1      : {result['val_macro_f1']:.4f}")
    print(f"Accuracy      : {result['val_acc']:.4f}")
    print(f"Negative Recall: {result['neg_recall']:.4f}")
    print(f"Negative F1   : {result['neg_f1']:.4f}")
    print(f"Training time : {result['minutes']:.1f} min")

    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()

    return result

In [17]:
if RUN_MANUAL_WEIGHT_EXPERIMENTS:
    result_15 = train_manual_weight_run(
        tag='distilbert_manual_weight_1p5',
        negative_weight=1.5,
        positive_weight=1.0
    )
    display(result_15)
else:
    print('Skipping manual weight run 1.5x (RUN_MANUAL_WEIGHT_EXPERIMENTS=False)')


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



=== distilbert_manual_weight_1p5 ===
Class weights [negative, positive]: [1.5, 1.0]


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Neg Precision,Neg Recall,Neg F1,Pos Precision,Pos Recall,Pos F1
1,0.127326,0.126331,0.967690,0.938392,0.906128,0.885915,0.895907,0.978849,0.982914,0.980877
2,0.076705,0.133837,0.969291,0.942261,0.897355,0.908222,0.902756,0.982875,0.980660,0.981766


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Validation result:
Macro-F1      : 0.9423
Accuracy      : 0.9693
Negative Recall: 0.9082
Negative F1   : 0.9028
Training time : 5.5 min


{'run': 'distilbert_manual_weight_1p5',
 'change': 'manual [1.5:1]',
 'lr': 2e-05,
 'batch': 16,
 'epochs': 2,
 'seed': 42,
 'minutes': 5.5,
 'best_epoch': 2,
 'val_macro_f1': 0.9423,
 'val_acc': 0.9693,
 'neg_recall': 0.9082,
 'neg_f1': 0.9028,
 'val_loss_by_epoch': [0.1263, 0.1338]}

In [18]:
if RUN_MANUAL_WEIGHT_EXPERIMENTS:
    result_20 = train_manual_weight_run(
        tag='distilbert_manual_weight_2x',
        negative_weight=2.0,
        positive_weight=1.0
    )
    display(result_20)
else:
    print('Skipping manual weight run 2.0x (RUN_MANUAL_WEIGHT_EXPERIMENTS=False)')


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



=== distilbert_manual_weight_2x ===
Class weights [negative, positive]: [2.0, 1.0]


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Neg Precision,Neg Recall,Neg F1,Pos Precision,Pos Recall,Pos F1
1,0.141938,0.140729,0.968090,0.938962,0.910644,0.883365,0.896797,0.978407,0.983863,0.981128
2,0.080094,0.150740,0.969591,0.942869,0.897549,0.910134,0.903797,0.983226,0.980660,0.981941


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Validation result:
Macro-F1      : 0.9429
Accuracy      : 0.9696
Negative Recall: 0.9101
Negative F1   : 0.9038
Training time : 5.2 min


{'run': 'distilbert_manual_weight_2x',
 'change': 'manual [2.0:1]',
 'lr': 2e-05,
 'batch': 16,
 'epochs': 2,
 'seed': 42,
 'minutes': 5.2,
 'best_epoch': 2,
 'val_macro_f1': 0.9429,
 'val_acc': 0.9696,
 'neg_recall': 0.9101,
 'neg_f1': 0.9038,
 'val_loss_by_epoch': [0.1407, 0.1507]}

In [19]:
if RUN_MANUAL_WEIGHT_EXPERIMENTS:
    result_30 = train_manual_weight_run(
        tag='distilbert_manual_weight_3x',
        negative_weight=3.0,
        positive_weight=1.0
    )
    display(result_30)
else:
    print('Skipping manual weight run 3.0x (RUN_MANUAL_WEIGHT_EXPERIMENTS=False)')


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



=== distilbert_manual_weight_3x ===
Class weights [negative, positive]: [3.0, 1.0]


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Neg Precision,Neg Recall,Neg F1,Pos Precision,Pos Recall,Pos F1
1,0.163851,0.175105,0.967390,0.937174,0.915719,0.872530,0.893603,0.976476,0.985050,0.980744
2,0.094947,0.175420,0.970491,0.944489,0.901639,0.911409,0.906498,0.983474,0.981490,0.982481


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Validation result:
Macro-F1      : 0.9445
Accuracy      : 0.9705
Negative Recall: 0.9114
Negative F1   : 0.9065
Training time : 5.1 min


{'run': 'distilbert_manual_weight_3x',
 'change': 'manual [3.0:1]',
 'lr': 2e-05,
 'batch': 16,
 'epochs': 2,
 'seed': 42,
 'minutes': 5.1,
 'best_epoch': 2,
 'val_macro_f1': 0.9445,
 'val_acc': 0.9705,
 'neg_recall': 0.9114,
 'neg_f1': 0.9065,
 'val_loss_by_epoch': [0.1751, 0.1754]}

In [20]:
if RUN_MANUAL_WEIGHT_EXPERIMENTS:
    result_40 = train_manual_weight_run(
        tag='distilbert_manual_weight_4x',
        negative_weight=4.0,
        positive_weight=1.0
    )
    display(result_40)
else:
    print('Skipping manual weight run 4.0x (RUN_MANUAL_WEIGHT_EXPERIMENTS=False)')


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



=== distilbert_manual_weight_4x ===
Class weights [negative, positive]: [4.0, 1.0]


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Neg Precision,Neg Recall,Neg F1,Pos Precision,Pos Recall,Pos F1
1,0.172361,0.197779,0.967190,0.937224,0.907955,0.880178,0.893851,0.977820,0.983389,0.980596
2,0.101034,0.191105,0.969391,0.942729,0.892968,0.914595,0.903652,0.984029,0.979592,0.981805


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Validation result:
Macro-F1      : 0.9427
Accuracy      : 0.9694
Negative Recall: 0.9146
Negative F1   : 0.9037
Training time : 4.9 min


{'run': 'distilbert_manual_weight_4x',
 'change': 'manual [4.0:1]',
 'lr': 2e-05,
 'batch': 16,
 'epochs': 2,
 'seed': 42,
 'minutes': 4.9,
 'best_epoch': 2,
 'val_macro_f1': 0.9427,
 'val_acc': 0.9694,
 'neg_recall': 0.9146,
 'neg_f1': 0.9037,
 'val_loss_by_epoch': [0.1978, 0.1911]}

In [21]:
if RUN_MANUAL_WEIGHT_EXPERIMENTS:
    result_50 = train_manual_weight_run(
        tag='distilbert_manual_weight_5x',
        negative_weight=5.0,
        positive_weight=1.0
    )
    display(result_50)
else:
    print('Skipping manual weight run 5.0x (RUN_MANUAL_WEIGHT_EXPERIMENTS=False)')


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



=== distilbert_manual_weight_5x ===
Class weights [negative, positive]: [5.0, 1.0]


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Neg Precision,Neg Recall,Neg F1,Pos Precision,Pos Recall,Pos F1
1,0.181793,0.214926,0.968090,0.938637,0.916112,0.876992,0.896125,0.977281,0.985050,0.981150
2,0.104875,0.200431,0.969191,0.942325,0.892835,0.913321,0.902962,0.983794,0.979592,0.981688


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Validation result:
Macro-F1      : 0.9423
Accuracy      : 0.9692
Negative Recall: 0.9133
Negative F1   : 0.9030
Training time : 4.7 min


{'run': 'distilbert_manual_weight_5x',
 'change': 'manual [5.0:1]',
 'lr': 2e-05,
 'batch': 16,
 'epochs': 2,
 'seed': 42,
 'minutes': 4.7,
 'best_epoch': 2,
 'val_macro_f1': 0.9423,
 'val_acc': 0.9692,
 'neg_recall': 0.9133,
 'neg_f1': 0.903,
 'val_loss_by_epoch': [0.2149, 0.2004]}

In [22]:
# Final full-training experiment:
# 281,271 reviews + 4x negative class weight

full_train_tokenized = tokenize(full_tr)

print("Tokenized full training set:", len(full_train_tokenized))
print("Validation set:", len(tok_val))

Map:   0%|          | 0/281271 [00:00<?, ? examples/s]

Tokenized full training set: 281271
Validation set: 9997


In [24]:
train_dataset = full_train_tokenized
learning_rate = 2e-5
batch_size = 16
epochs = 2
negative_weight = 4.0
positive_weight = 1.0

In [26]:
# Exploratory 50k training with 3x negative class weight
WEIGHTED_50K_TAG = 'distilbert_50k_manual_3x'

if RUN_MANUAL_WEIGHT_EXPERIMENTS:
    weighted_50k_result = train_manual_weight_run(
        tag=WEIGHTED_50K_TAG,
        negative_weight=3.0,
        positive_weight=1.0
    )
    display(weighted_50k_result)
else:
    print('Skipping 50k 3x manual weighted training run (RUN_MANUAL_WEIGHT_EXPERIMENTS=False)')


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



=== distilbert_full281k_manual_4x ===
Class weights [negative, positive]: [3.0, 1.0]


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Neg Precision,Neg Recall,Neg F1,Pos Precision,Pos Recall,Pos F1
1,0.164496,0.175345,0.967490,0.937316,0.916890,0.871893,0.893826,0.976367,0.985287,0.980807
2,0.094918,0.176215,0.970391,0.944258,0.902085,0.910134,0.906091,0.983242,0.981609,0.982425


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Validation result:
Macro-F1      : 0.9443
Accuracy      : 0.9704
Negative Recall: 0.9101
Negative F1   : 0.9061
Training time : 5.0 min


{'run': 'distilbert_full281k_manual_4x',
 'change': 'manual [3.0:1]',
 'lr': 2e-05,
 'batch': 16,
 'epochs': 2,
 'seed': 42,
 'minutes': 5.0,
 'best_epoch': 2,
 'val_macro_f1': 0.9443,
 'val_acc': 0.9704,
 'neg_recall': 0.9101,
 'neg_f1': 0.9061,
 'val_loss_by_epoch': [0.1753, 0.1762]}

In [27]:
LOGITS_FILE = ART / "distilbert281k_test_logits.npy"

In [28]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

test_logits = np.load(LOGITS_FILE)
y_test = test["label"].to_numpy()

y_pred = test_logits.argmax(axis=1)

probs = np.exp(test_logits - test_logits.max(axis=1, keepdims=True))
probs = probs / probs.sum(axis=1, keepdims=True)

print("Test samples:", len(y_test))
print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=["negative", "positive"],
        digits=4
    )
)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Macro-F1:", f1_score(y_test, y_pred, average="macro"))
print("ROC-AUC:", roc_auc_score(y_test, probs[:, 1]))

tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
print(f"\nTN={tn:,}  FP={fp:,} | FN={fn:,}  TP={tp:,}")

Test samples: 72846

Classification Report:
              precision    recall  f1-score   support

    negative     0.9186    0.9138    0.9162     11372
    positive     0.9841    0.9850    0.9845     61474

    accuracy                         0.9739     72846
   macro avg     0.9513    0.9494    0.9504     72846
weighted avg     0.9739    0.9739    0.9739     72846

Accuracy: 0.9739038519616726
Macro-F1: 0.9503728636060589
ROC-AUC: 0.9930409426684864

TN=10,392  FP=980 | FN=921  TP=60,553


In [34]:
# ============================================================
# EVALUATION - DISTILBERT 3x NEGATIVE CLASS WEIGHT (50k RUN)
# ============================================================

import numpy as np
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
from datasets import Dataset as HFDataset
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

if RUN_MANUAL_WEIGHT_EXPERIMENTS:
    # 1. Load the saved 3x manual model checkpoint from 50k sweep
    THREE_X_DIR = RUNS / "distilbert_50k_manual_3x" / "checkpoint-6250"
    print("Loading 3x model from:", THREE_X_DIR)

    tokenizer_3x = AutoTokenizer.from_pretrained("distilbert-base-uncased")
    model_3x = AutoModelForSequenceClassification.from_pretrained(THREE_X_DIR)

    # 2. Prepare the untouched test set
    test_ds_3x = HFDataset.from_pandas(test.reset_index(drop=True)).map(
        lambda batch: tokenizer_3x(batch["text"], truncation=True, max_length=MAX_LEN),
        batched=True,
        batch_size=2000,
        remove_columns=["text"]
    )
    print(f"\nTest samples: {len(test_ds_3x):,}")

    # 3. Run inference
    trainer_3x = Trainer(
        model=model_3x,
        data_collator=DataCollatorWithPadding(tokenizer_3x),
        args=TrainingArguments(
            output_dir=str(RUNS / "eval_3x"),
            per_device_eval_batch_size=64,
            fp16=torch.cuda.is_available(),
            report_to="none"
        )
    )

    output_3x = trainer_3x.predict(test_ds_3x)
    logits_3x = np.asarray(output_3x.predictions, dtype=np.float64)
    y_test_3x = np.asarray(output_3x.label_ids)
    y_pred_3x = logits_3x.argmax(axis=1)

    # 4. Probabilities
    shifted_logits = logits_3x - logits_3x.max(axis=1, keepdims=True)
    exp_logits = np.exp(shifted_logits)
    probs_3x = exp_logits / exp_logits.sum(axis=1, keepdims=True)
    positive_probs_3x = probs_3x[:, 1]

    # 5. Metrics
    accuracy_3x = accuracy_score(y_test_3x, y_pred_3x)
    macro_f1_3x = f1_score(y_test_3x, y_pred_3x, average="macro")
    roc_auc_3x = roc_auc_score(y_test_3x, positive_probs_3x)
    report_3x = classification_report(y_test_3x, y_pred_3x, target_names=["negative", "positive"], digits=4)
    tn_3x, fp_3x, fn_3x, tp_3x = (int(v) for v in confusion_matrix(y_test_3x, y_pred_3x).ravel())

    # 6. Display results
    print("\n" + "=" * 60)
    print("DISTILBERT 3x NEGATIVE CLASS WEIGHT (50k RUN) TEST EVALUATION")
    print("=" * 60)
    print(f"Accuracy: {accuracy_3x:.4f} | Macro-F1: {macro_f1_3x:.4f} | ROC-AUC: {roc_auc_3x:.4f}")
    print(f"TN={tn_3x:,}  FP={fp_3x:,} | FN={fn_3x:,}  TP={tp_3x:,}")
    print("=" * 60)
else:
    print("Skipping 3x manual weight test evaluation (RUN_MANUAL_WEIGHT_EXPERIMENTS=False)")



Loading 4× model from:
A:\DS-Review-AI\scratch\runs\distilbert_full281k_manual_4x\checkpoint-6250


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Map:   0%|          | 0/72846 [00:00<?, ? examples/s]


Test samples: 72,846




DISTILBERT — 4× NEGATIVE CLASS WEIGHT
FINAL 72,846-REVIEW TEST SET

Accuracy:        0.9672
Macro-F1:        0.9378
ROC-AUC:         0.9899

Classification Report:
              precision    recall  f1-score   support

    negative     0.8939    0.8962    0.8951     11372
    positive     0.9808    0.9803    0.9806     61474

    accuracy                         0.9672     72846
   macro avg     0.9373    0.9383    0.9378     72846
weighted avg     0.9672    0.9672    0.9672     72846

Confusion Matrix:
TN = 10,192   FP = 1,180   FN = 1,210   TP = 60,264
